# 03 — CSV vs JSON vs Parquet

## Learning objectives
- Understand why data formats exist and how their trade-offs differ.
- Compare CSV, JSONL, and Parquet on the same dataset.
- Connect format choice to real pipeline workloads.

## The minimum theory

A **data format** is a contract for how values are stored and reconstructed.

- **CSV:** simple tabular text; portable, but types are weakly represented.
- **JSONL:** one record per line; flexible and common at API/event boundaries.
- **Parquet:** typed, compressed, columnar storage designed for analytical workloads.

The important lesson is not “one format always wins.” Format choice depends on who consumes the data, whether it is a system boundary or analytical dataset, and what reads the data repeatedly.


## Capability contract

**Capability:** C01 — System Foundations  
**Workstream:** Foundation  
**Primary roles:** Data Engineer|Analytics Engineer  
**You will prove:** Explain the mechanism and reproduce it with a deterministic experiment.


## 🖊️ Sketch note

![Sketch note — CSV vs JSON vs Parquet](../assets/sketch-notes/03-csv-vs-json-vs-parquet.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S02 — API schema drift (DATA-1051)** → [`../work_simulations/02_acme_commerce_api_schema_drift/README.md`](../work_simulations/02_acme_commerce_api_schema_drift/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Predict before you run

Write one prediction and the mechanism behind it before executing the next experiment.


## Production bridge

State what changes when this mechanism is used at production scale: reliability, concurrency, data volume, observability, security, cost or recovery.


## Diagnose

If a format measurement surprises you, check file size, compression, columns selected and read pattern before claiming one format is universally faster.


In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("pyarrow") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","pyarrow>=16,<22"])
from pathlib import Path
import numpy as np, pandas as pd, time
rng=np.random.default_rng(42)
df=pd.DataFrame({"event_id":np.arange(100_000,dtype=np.int64),"customer_id":rng.integers(1,10_000,100_000),"country":rng.choice(["IN","US","DE","GB","SG"],100_000),"event_type":rng.choice(["view","click","purchase"],100_000),"amount":np.round(rng.gamma(2,50,100_000),2),"description":rng.choice(["landing","product","checkout","support","pricing"],100_000)})
out=Path("data"); out.mkdir(exist_ok=True)
paths={"CSV":out/"events.csv","JSONL":out/"events.jsonl","Parquet":out/"events.parquet"}
df.to_csv(paths["CSV"],index=False); df.to_json(paths["JSONL"],orient="records",lines=True); df.to_parquet(paths["Parquet"],index=False)


## Experiment — size and read workload

We measure full reads and a narrow read. The narrow read is important because columnar formats can avoid loading unused columns.

Treat this as an experiment, not a universal benchmark: hardware, libraries, compression, and data shape all matter.


In [ ]:
def timed(fn):
    best=float("inf")
    for _ in range(3):
        t=time.perf_counter(); fn(); best=min(best,time.perf_counter()-t)
    return best
results=[]
results.append(["CSV",paths["CSV"].stat().st_size,timed(lambda:pd.read_csv(paths["CSV"])),timed(lambda:pd.read_csv(paths["CSV"],usecols=["customer_id","amount"]))])
results.append(["JSONL",paths["JSONL"].stat().st_size,timed(lambda:pd.read_json(paths["JSONL"],lines=True)),None])
results.append(["Parquet",paths["Parquet"].stat().st_size,timed(lambda:pd.read_parquet(paths["Parquet"])),timed(lambda:pd.read_parquet(paths["Parquet"],columns=["customer_id","amount"]))])
pd.DataFrame(results,columns=["format","bytes","full_read_seconds","narrow_read_seconds"])


## Your task

**Goal:** Choose the right file format for different workloads.

**Do this:** Implement `choose_format(consumer, api_boundary, analytical_reads)` so it returns both a recommendation and the reasoning. Test it against more than one workload.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail


### Engineering challenge
Write `choose_format(consumer, api_boundary, analytical_reads)` that returns a recommendation **and the reasoning**.

**Next:** Arrow and columnar memory.


## Break — workload changes the conclusion

In [ ]:
# Narrow analytical reads favour formats that can avoid loading unused columns.
# Add a large text column and compare the same narrow read again.
df["notes"]="x"*200
print("columns now:",len(df.columns),"narrow columns requested:",2)
assert len(df.columns)>4

---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 03](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/03.md)
